# 07 — Pooled Yoko + Yangambi: Musanga fractional cover

Clean benchmark notebook.

**Question:** how well can observation-year satellite information predict UAV-derived fractional Musanga cover, and how does performance change from 10 to 50 m support?

Current scope:
- Yoko (2020) + YB_Carlier / Yangambi (2024), pooled as one modelling dataset.
- Mos is not loaded or modified.
- Fractional cover is calculated directly from native binary UAV pixels at 10, 20, 30 and 50 m.
- Predictor sets use **T0 only**: observation-year Google Satellite Embeddings, observation-year quarterly Sentinel-2, and their fusion.
- No T−1 or T+1 imagery is used; there is no post-observation information leakage.
- Simple site-stratified 80/20 development split.
- Continuous and detection metrics are reported.

Spatial-block validation is deliberately left out until the basic signal, predictor contribution and scale dependence are clear.

## 01 — Setup

In [ ]:
!pip -q install earthengine-api geemap rasterio scikit-learn scipy joblib

from pathlib import Path
import re, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, balanced_accuracy_score
)
import ee, geemap

from google.colab import drive
DRIVE_ROOT=Path("/content/drive")
MYDRIVE=DRIVE_ROOT/"MyDrive"
if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Google Drive already mounted.")

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"
UAV_DIR=ROOT/"data"/"multisite_uav"
OUT=ROOT/"work"/"07_pooled_yoko_yangambi"
OLD_OUT=ROOT/"work"/"multisite_embedding_fractional_cover"
OUT.mkdir(parents=True,exist_ok=True)

EE_PROJECT="tropsedslu"
ACTIVE_SITES={"Yoko":2020,"YB_Carlier":2024}
SUPPORTS=[10,20,30,50]
MIN_UAV_COVERAGE=0.90
RANDOM_SEED=42
TEST_SIZE=0.20
N_TREES=400
FORCE=False
COVER_THRESHOLDS=[0.01,0.05,0.20,0.50]

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Output:",OUT)

## 02 — Load only Yoko + Yangambi UAV masks

In [ ]:
rows=[]
for site,year in ACTIVE_SITES.items():
    hits=sorted(UAV_DIR.glob(f"{site}_musanga_binary_mask_threshold_*.tif"))
    if len(hits)!=1:
        raise RuntimeError(f"{site}: expected one binary mask, found {len(hits)}")
    p=hits[0]
    m=re.search(r"threshold_([0-9.]+)\.tif$",p.name)
    with rasterio.open(p) as ds:
        b=transform_bounds(ds.crs,"EPSG:4326",*ds.bounds,densify_pts=21)
        rows.append({
            "site":site,"obs_year":year,"threshold":float(m.group(1)),
            "path":str(p),"uav_res_m":abs(ds.transform.a),
            "lon_min":b[0],"lat_min":b[1],"lon_max":b[2],"lat_max":b[3]
        })

inventory=pd.DataFrame(rows)
display(inventory)
assert set(inventory.site)==set(ACTIVE_SITES)
print("Only Yoko + Yangambi are active. Mos data remain untouched.")

## 03 — Satellite predictor caches

For each site, satellite predictors are restricted to the **UAV observation year (T0)**:

- **Embedding:** annual Google Satellite Embedding for T0 only (64 predictors).
- **Sentinel-2:** four T0 quarterly composites with 10 reflectance bands + NDVI, NDMI, NBR and red-edge NDVI (56 predictors).
- **Fusion:** all 120 predictors.

This keeps the comparison temporally consistent across Yoko (2020) and Yangambi (2024) and avoids using post-observation imagery. Existing compatible T0 embedding caches are reused. T0-only Sentinel-2 products use distinct cache names.

In [ ]:
EMB_BANDS=[f"A{i:02d}" for i in range(64)]
S2_BANDS=["B2","B3","B4","B5","B6","B7","B8","B8A","B11","B12"]
S2_FEATURES=S2_BANDS+["NDVI","NDMI","NBR","RENDVI"]

def roi_for_mask(mask_path):
    with rasterio.open(mask_path) as ds:
        b=transform_bounds(ds.crs,"EPSG:4326",*ds.bounds,densify_pts=21)
    return ee.Geometry.Rectangle(list(b),geodesic=False)

def valid_tif(path,count=None):
    try:
        with rasterio.open(path) as ds:
            return ds.width>0 and ds.height>0 and (count is None or ds.count==count)
    except Exception:
        return False

def embedding_path(site,year):
    old=OLD_OUT/"embeddings"/f"{site}_embedding_{year}_10m.tif"
    new=OUT/"embeddings"/f"{site}_embedding_{year}_10m.tif"
    if old.exists() and valid_tif(old,64): return old
    return new

def get_embedding(site,mask_path,year):
    out=embedding_path(site,year)
    out.parent.mkdir(parents=True,exist_ok=True)
    if out.exists() and valid_tif(out,64) and not FORCE:
        print("Reusing",out.name)
        return out
    roi=roi_for_mask(mask_path)
    col=(ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
         .filterDate(f"{year}-01-01",f"{year+1}-01-01").filterBounds(roi))
    if col.size().getInfo()==0:
        raise RuntimeError(f"No embedding for {site} {year}")
    print("Downloading embedding",site,year)
    geemap.ee_export_image(
        col.mosaic().select(EMB_BANDS),filename=str(out),
        scale=10,region=roi,file_per_band=False
    )
    if not valid_tif(out,64):
        raise RuntimeError(f"Embedding export failed: {out}")
    return out

def mask_s2(img):
    scl=img.select("SCL")
    clear=(scl.neq(0).And(scl.neq(1)).And(scl.neq(3))
           .And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))
    r=img.select(S2_BANDS).multiply(0.0001)
    out=(r.addBands([
        r.normalizedDifference(["B8","B4"]).rename("NDVI"),
        r.normalizedDifference(["B8","B11"]).rename("NDMI"),
        r.normalizedDifference(["B8","B12"]).rename("NBR"),
        r.normalizedDifference(["B8A","B5"]).rename("RENDVI")
    ]).updateMask(clear))
    # Critical: preserve acquisition time so filterDate still works after map(mask_s2).
    return out.copyProperties(img, ["system:time_start", "system:index"])

def s2_path(site,year):
    return OUT/"sentinel2"/f"{site}_S2_quarterly_T0_{year}_10m.tif"

def get_s2(site,mask_path,year):
    out=s2_path(site,year)
    out.parent.mkdir(parents=True,exist_ok=True)
    if out.exists() and valid_tif(out,56) and not FORCE:
        print("Reusing",out.name)
        return out
    roi=roi_for_mask(mask_path)
    # Same ordering/logic as the working Notebook 06 M05 implementation.
    # Filter the source collection first, then map the cloud-mask function.
    col=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi)
         .filterDate(f"{year}-01-01",f"{year+1}-01-01")
         .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE",80)).map(mask_s2))
    print(f"{site} {year}: {col.size().getInfo()} total candidate S2 scenes")
    ims=[]
    for yy in [year]:
        for q,month in enumerate([1,4,7,10],1):
            qcol=col.filterDate(ee.Date.fromYMD(yy,month,1),ee.Date.fromYMD(yy,month,1).advance(3,"month"))
            n=qcol.size().getInfo()
            print(f"{site} {yy} Q{q}: {n} S2 scenes")
            if n==0:
                raise RuntimeError(f"No S2 scenes for {site} {yy} Q{q}")
            names=[f"S2_{yy}_Q{q}_{b}" for b in S2_FEATURES]
            ims.append(qcol.median().select(S2_FEATURES).rename(names))
    stack=ee.Image.cat(ims).toFloat()
    print("Downloading S2",site,year)
    geemap.ee_export_image(stack,filename=str(out),scale=10,region=roi,file_per_band=False)
    if not valid_tif(out,56):
        raise RuntimeError(f"S2 export failed: {out}")
    return out

satellite={}
for row in inventory.itertuples(index=False):
    mask=Path(row.path); y=int(row.obs_year)
    satellite[row.site]={
        "emb":get_embedding(row.site,mask,y),
        "s2":get_s2(row.site,mask,y)
    }

print("Satellite caches ready.")

## 04 — Native UAV fractional cover + predictors at 10/20/30/50 m

Fractional cover is always calculated **directly from the native binary UAV mask** at the requested support. It is never produced by averaging an existing 10 m fractional-cover target.

All satellite rasters are aligned and averaged directly onto that same target grid.

In [ ]:
# Fast table builder: align each satellite stack ONCE at 10 m per site, then aggregate
# the in-memory 10 m cubes for 20/30/50 m. The UAV target is still recomputed directly
# from the native binary UAV mask at every support (never from the 10 m target).

EMB_COLS=[f"E_T0_A{j:02d}" for j in range(64)]
S2_COLS=[f"S2_{j:03d}" for j in range(56)]
ALL_FEATURE_COLS=EMB_COLS+S2_COLS

def target_grid(ref_path,support):
    with rasterio.open(ref_path) as ref:
        factor=int(round(support/10))
        h=int(np.ceil(ref.height/factor)); w=int(np.ceil(ref.width/factor))
        tr=ref.transform*rasterio.Affine.scale(factor,factor)
        return (h,w),ref.crs,tr

def aggregate_native_uav(mask_path,shape,crs,tr):
    # Direct native-UAV numerator and denominator aggregation.
    # This avoids using 0 as NoData and avoids the old temporary validity GeoTIFF.
    with rasterio.open(mask_path) as src:
        a=src.read(1)
        nd=src.nodata
        valid=np.isfinite(a)
        if nd is not None:
            valid &= (a != nd)
        musanga=(valid & (a==1)).astype("float32")
        valid_f=valid.astype("float32")

        num=np.zeros(shape,np.float32)
        den=np.zeros(shape,np.float32)
        reproject(
            musanga,num,
            src_transform=src.transform,src_crs=src.crs,src_nodata=None,
            dst_transform=tr,dst_crs=crs,dst_nodata=0.0,
            resampling=Resampling.average,init_dest_nodata=True
        )
        reproject(
            valid_f,den,
            src_transform=src.transform,src_crs=src.crs,src_nodata=None,
            dst_transform=tr,dst_crs=crs,dst_nodata=0.0,
            resampling=Resampling.average,init_dest_nodata=True
        )
        frac=np.full(shape,np.nan,np.float32)
        np.divide(num,den,out=frac,where=den>0)
        return frac,den

def align_stack_once(path,ref_path):
    # Exact 10 m reference grid; one reprojection per source band, once per site.
    with rasterio.open(ref_path) as ref:
        shape=(ref.height,ref.width); crs=ref.crs; tr=ref.transform
    with rasterio.open(path) as src:
        out=np.full((src.count,*shape),np.nan,np.float32)
        same=(src.crs==crs and src.transform.almost_equals(tr)
              and (src.height,src.width)==shape)
        if same:
            out=src.read().astype("float32")
            valid=np.isfinite(out)
            if src.nodata is not None:
                valid &= (out != src.nodata)
            out[~valid]=np.nan
        else:
            for b in range(src.count):
                reproject(
                    rasterio.band(src,b+1),out[b],
                    src_transform=src.transform,src_crs=src.crs,src_nodata=src.nodata,
                    dst_transform=tr,dst_crs=crs,dst_nodata=np.nan,
                    resampling=Resampling.bilinear,init_dest_nodata=True
                )
        return out

def aggregate_cube_from_10m(cube,factor,out_shape):
    if factor==1:
        return cube
    bands,h,w=cube.shape
    oh,ow=out_shape
    ph=oh*factor; pw=ow*factor
    padded=np.full((bands,ph,pw),np.nan,np.float32)
    padded[:,:h,:w]=cube
    z=padded.reshape(bands,oh,factor,ow,factor)
    count=np.sum(np.isfinite(z),axis=(2,4))
    total=np.nansum(z,axis=(2,4))
    out=np.full((bands,oh,ow),np.nan,np.float32)
    np.divide(total,count,out=out,where=count>0)
    return out

# Prepare expensive satellite alignment only once per site.
satellite_10m={}
for row in inventory.itertuples(index=False):
    site=row.site
    print(f"Preparing 10 m satellite cube once: {site}")
    emb=align_stack_once(satellite[site]["emb"],satellite[site]["emb"])
    s2=align_stack_once(satellite[site]["s2"],satellite[site]["emb"])
    if emb.shape[0]!=64 or s2.shape[0]!=56:
        raise RuntimeError(f"{site}: expected 64 embedding + 56 S2 bands, got {emb.shape[0]} + {s2.shape[0]}")
    satellite_10m[site]=np.concatenate([emb,s2],axis=0)
    print(f"  {satellite_10m[site].shape[0]} bands ready")

def build_table_fast(row,support):
    site=row.site; mask=Path(row.path); y=int(row.obs_year)
    ref=satellite[site]["emb"]
    shape,crs,tr=target_grid(ref,support)
    frac,cov=aggregate_native_uav(mask,shape,crs,tr)

    factor=int(round(support/10))
    cube=aggregate_cube_from_10m(satellite_10m[site],factor,shape)

    # A modelling row exists only if target, coverage and ALL 120 predictors are valid.
    good=(np.isfinite(frac)&(frac>=0)&(frac<=1)
          &(cov>=MIN_UAV_COVERAGE)
          &np.all(np.isfinite(cube),axis=0))
    rr,cc=np.where(good)
    xs,ys=rasterio.transform.xy(tr,rr,cc,offset="center")

    base={
        "site":np.repeat(site,len(rr)),"obs_year":np.repeat(y,len(rr)),
        "support_m":np.repeat(support,len(rr)),"row":rr,"col":cc,
        "x":np.asarray(xs),"y":np.asarray(ys),
        "uav_valid_fraction":cov[good],"musanga_fraction":frac[good]
    }
    feats={name:cube[j][good] for j,name in enumerate(ALL_FEATURE_COLS)}
    return pd.DataFrame({**base,**feats})

tables={}
for support in SUPPORTS:
    parts=[]
    print(f"\n=== {support} m ===")
    for row in inventory.itertuples(index=False):
        cache=OUT/"tables_T0"/f"{row.site}_{support}m.csv"
        cache.parent.mkdir(parents=True,exist_ok=True)
        if cache.exists() and not FORCE:
            print("Reusing",cache.name)
            d=pd.read_csv(cache)
        else:
            print("Building",row.site)
            d=build_table_fast(row,support)
            d.to_csv(cache,index=False)

        # Hard QA gate applies equally to old cached and newly built tables.
        valid=(np.isfinite(d["musanga_fraction"].to_numpy(float))
               & d["musanga_fraction"].between(0,1).to_numpy()
               & np.isfinite(d["uav_valid_fraction"].to_numpy(float))
               & (d["uav_valid_fraction"].to_numpy(float)>=MIN_UAV_COVERAGE)
               & np.isfinite(d[ALL_FEATURE_COLS].to_numpy(float)).all(axis=1))
        n_bad=int((~valid).sum())
        if n_bad:
            print(f"  Excluding {n_bad:,} no-data/invalid rows before modelling")
            d=d.loc[valid].reset_index(drop=True)
        assert len(d)>0, f"No valid modelling rows for {row.site} at {support} m"
        assert np.isfinite(d[ALL_FEATURE_COLS+["musanga_fraction","uav_valid_fraction"]].to_numpy(float)).all()
        parts.append(d)

    tables[support]=pd.concat(parts,ignore_index=True)
    display(tables[support].groupby("site").musanga_fraction.agg(["count","mean","median","max"]))

# Free the large aligned cubes after tables are ready.
satellite_10m.clear()
print("\nFast table preparation complete.")


## 05 — Simple pooled 80/20 benchmark

At each support we compare three predictor sets on the **same held-out cells**:

1. T0 embeddings only (64 predictors);
2. T0 Sentinel-2 quarterly temporal/spectral features only (56 predictors);
3. T0 embeddings + Sentinel-2 fusion (120 predictors).

The split is stratified by site so Yoko and Yangambi are represented in both training and test data. This is a development benchmark, not spatially independent validation.
**QA/persistence:** rows with UAV no-data, <90% valid UAV coverage, or any non-finite satellite predictor are excluded before splitting or fitting. Each fitted RF and its held-out predictions are saved to Drive for reuse/reproducibility.


In [ ]:
EMB_COLS=[f"E_T0_A{j:02d}" for j in range(64)]
S2_COLS=[f"S2_{j:03d}" for j in range(56)]
FEATURE_SETS={"Embeddings":EMB_COLS,"Sentinel2":S2_COLS,"Fusion":EMB_COLS+S2_COLS}

def metrics(y,p):
    return {
        "R2":r2_score(y,p),
        "RMSE":mean_squared_error(y,p)**0.5,
        "MAE":mean_absolute_error(y,p),
        "Spearman":spearmanr(y,p).statistic,
        "bias":float(np.mean(p-y)),
        "obs_mean":float(np.mean(y)),
        "pred_mean":float(np.mean(p))
    }

continuous=[]; detection=[]; heldout={}
MODELS=OUT/"models_T0"; MODELS.mkdir(parents=True,exist_ok=True)
PREDICTIONS=OUT/"predictions_T0"; PREDICTIONS.mkdir(parents=True,exist_ok=True)

for support,d0 in tables.items():
    # Final hard modelling gate. No UAV nodata, insufficient UAV coverage,
    # NaN/Inf satellite predictors, or invalid target values can reach RF.fit/predict.
    required=EMB_COLS+S2_COLS+["musanga_fraction","uav_valid_fraction"]
    valid=(np.isfinite(d0[required].to_numpy(float)).all(axis=1)
           & d0["musanga_fraction"].between(0,1).to_numpy()
           & (d0["uav_valid_fraction"].to_numpy(float)>=MIN_UAV_COVERAGE))
    d=d0.loc[valid].reset_index(drop=True)
    excluded=int(len(d0)-len(d))
    print(f"\n{support} m: {len(d):,} valid modelling cells; {excluded:,} excluded by no-data/coverage gate")
    assert len(d)>0
    assert np.isfinite(d[required].to_numpy(float)).all()

    idx=np.arange(len(d))
    tr,te=train_test_split(idx,test_size=TEST_SIZE,random_state=RANDOM_SEED,
                           stratify=d.site.to_numpy())
    ytr=d.iloc[tr].musanga_fraction.to_numpy(np.float32)
    yte=d.iloc[te].musanga_fraction.to_numpy(np.float32)

    for name,cols in FEATURE_SETS.items():
        Xtr=d.iloc[tr][cols].to_numpy(np.float32)
        Xte=d.iloc[te][cols].to_numpy(np.float32)
        assert np.isfinite(Xtr).all() and np.isfinite(Xte).all()
        assert np.isfinite(ytr).all() and np.isfinite(yte).all()

        model_path=MODELS/f"RF_{support}m_{name}_T0.joblib"
        model=RandomForestRegressor(
            n_estimators=N_TREES,min_samples_leaf=2,max_features="sqrt",
            n_jobs=-1,random_state=RANDOM_SEED
        )
        model.fit(Xtr,ytr)
        joblib.dump({
            "model":model,
            "support_m":support,
            "predictors":name,
            "feature_columns":cols,
            "random_seed":RANDOM_SEED,
            "test_size":TEST_SIZE,
            "min_uav_coverage":MIN_UAV_COVERAGE,
            "n_train":len(tr),
            "n_test":len(te)
        },model_path,compress=3)

        pred=np.clip(model.predict(Xte),0,1)
        continuous.append({"support_m":support,"predictors":name,
                           "n_train":len(tr),"n_test":len(te),**metrics(yte,pred)})
        heldout[(support,name)]=(te,yte,pred)

        pred_df=d.iloc[te][["site","obs_year","support_m","row","col","x","y",
                            "uav_valid_fraction","musanga_fraction"]].copy()
        pred_df["prediction"]=pred
        pred_df.to_csv(PREDICTIONS/f"heldout_{support}m_{name}_T0.csv",index=False)

        for thr in COVER_THRESHOLDS:
            truth=(yte>thr).astype(int)
            hard=(pred>thr).astype(int)
            detection.append({
                "support_m":support,"predictors":name,"threshold":thr,
                "prevalence":truth.mean(),
                "ROC_AUC":roc_auc_score(truth,pred) if len(np.unique(truth))==2 else np.nan,
                "PR_AUC":average_precision_score(truth,pred) if len(np.unique(truth))==2 else np.nan,
                "F1":f1_score(truth,hard,zero_division=0),
                "precision":precision_score(truth,hard,zero_division=0),
                "recall":recall_score(truth,hard,zero_division=0),
                "balanced_accuracy":balanced_accuracy_score(truth,hard) if len(np.unique(truth))==2 else np.nan
            })

continuous=pd.DataFrame(continuous)
detection=pd.DataFrame(detection)

print("CONTINUOUS FRACTIONAL-COVER PERFORMANCE")
display(continuous.sort_values(["support_m","predictors"]))

print("DETECTION PERFORMANCE")
display(detection.sort_values(["threshold","support_m","predictors"]))

## 06 — Aggregation and predictor comparison

In [ ]:
for metric in ["R2","Spearman","RMSE","MAE"]:
    fig,ax=plt.subplots(figsize=(7,5))
    for name,z in continuous.groupby("predictors"):
        z=z.sort_values("support_m")
        ax.plot(z.support_m,z[metric],marker="o",label=name)
    ax.set_xlabel("Spatial support (m)")
    ax.set_ylabel(metric)
    ax.set_title(f"Pooled Yoko + Yangambi — {metric}")
    ax.legend(); ax.grid(alpha=.25)
    plt.show()

for metric in ["ROC_AUC","PR_AUC","F1"]:
    for thr in COVER_THRESHOLDS:
        fig,ax=plt.subplots(figsize=(7,5))
        z=detection[detection.threshold==thr]
        for name,q in z.groupby("predictors"):
            q=q.sort_values("support_m")
            ax.plot(q.support_m,q[metric],marker="o",label=name)
        ax.set_xlabel("Spatial support (m)")
        ax.set_ylabel(metric)
        ax.set_title(f"{metric} — Musanga cover > {thr:g}")
        ax.legend(); ax.grid(alpha=.25)
        plt.show()

RESULTS=OUT/"results"; RESULTS.mkdir(parents=True,exist_ok=True)
continuous.to_csv(RESULTS/"continuous_metrics.csv",index=False)
detection.to_csv(RESULTS/"detection_metrics.csv",index=False)
print("Saved:",RESULTS)

## 07 — Interpretation checkpoint

Use this notebook first to establish:
- whether aggregation improves fractional-cover prediction consistently across R², rank correlation and absolute errors;
- whether explicit Sentinel-2 temporal/spectral information improves over learned embeddings;
- whether fusion improves over either source alone;
- how detectability changes for low (>1%, >5%) and higher (>20%, >50%) Musanga cover.

Only after this simple benchmark is understood should stricter spatial-block validation be added as a robustness experiment.